# 1. نظرة عامة وإعداد البيئة

هذا دفتر خط الأساس المقبول، ذو **درجة Kaggle العامة 0.800 بحسب المستخدم**. الدرجة تاريخية؛ لا يحسبها هذا الدفتر. شغّل الخلايا بالترتيب أو اختر **Restart Kernel ثم Run All**. كل مهمة لها إعدادات وتحميل/تدريب وتنبؤ مستقل أدناه.

من PowerShell في جذر المستودع، مرة واحدة:
```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements-baseline.txt
```
اختر نواة `.venv/Scripts/python.exe`. ضع الملفات الرسمية الأربعة في `data/raw/`: `train.csv` و`test.csv` و`recordings.csv` و`sample_submission.csv`. المدخلات غير مرفوعة إلى Git. أصول T05 الاثنا عشر والجدول الزمني المجمد موجودة في `models/current_baseline/`؛ لا تحتاج تقديمًا قديمًا أو مجلد Downloads.

المفعّل: CatBoost لـB الجمعة، Ridge موسع لـG الجمعة، وHGB لـD السبت. بقية المهام كما في0.806؛ إعداده محفوظ في `configs/baseline_0806.json`. لا تدريب T05 أو رفع تلقائي.


In [ ]:
from pathlib import Path
import sys, json, time

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "t05_forecast_config.json").is_file() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("افتح الدفتر داخل مستودع madinah-ai-hackathon")
for folder in (ROOT, ROOT / "src", ROOT / ".vendor"):
    if folder.exists():
        sys.path.insert(0, str(folder))
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
import baseline_tasks as tasks
import baseline_workflow as workflow
from current_baseline import ASSETS

thread_limit = threadpool_limits(limits=4)
started = time.perf_counter()
protected = workflow.snapshot_protected()
DATA = ROOT / "data/raw"
OUTPUT = ROOT / "outputs/current_baseline_0800"
OUTPUT_NAME = "submission_current_baseline.csv"
VERIFY_ACCEPTED = True  # للمرشح: False + اسم ومجلد منفصلان، بعد التقييم
OUTPUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUTPUT:", OUTPUT)

# 2. تحميل البيانات والتحقق منها

المسارات مشتقة من جذر المستودع. نفحص نسخ الحزم وبصمات المدخلات والأصول المجمدة، ثم المخطط وتكرار المفاتيح وأهداف التقديم. الخلايا المحجوبة لا تدخل جدول المشاهدات. المصدر: `current_baseline.preflight` و`pipeline.load`.

بعد تعديل مصدر خوارزمية، لا تغير البصمة المقبولة لإخفاء اختلاف: احتفظ بوضع التحقق لإعادة الإنتاج، ثم استخدم وضع مرشح مستقل عند إجراء تعديل مقصود.

In [ ]:
project = workflow.load_project(DATA)
train, test, sample = project["train"], project["test"], project["sample"]
targets, feeds = project["targets"], project["feeds"]
print("train:", train.shape, "test:", test.shape, "submission IDs:", len(sample))
display(targets.groupby(["kind", "day"]).size().rename("targets").reset_index())
display(train.head(3))

# 3. المعالجة المشتركة والإعدادات

`pipeline.table` يحول القراءات المحررة إلى جداول دقيقة × كاميرا لكل تسجيل؛ الغياب يبقى NaN. المرجع الإنتاجي للمزج يأتي من العينة الرسمية. في التقييم يعاد حساب مرجع كل طية دون تاريخها؛ لذلك ليست الأرقام المحلية درجة Kaggle.

`baseline_tasks.py` يحتوي دوال الخوارزميات المشتركة مع التقييم، بينما `baseline_workflow.py` ينسق المدخلات وترتيب IDs وتنسيق CSV القديم فقط. بعض أجزاء المصدر مشتركة: `v2_hidden.fit` يخدم HIDDEN ومرجع HIDDEN-FUTURE؛ `v2_missing.interp_at` يخدم معايرة المانحين وHIDDEN. تعديلها قد يؤثر في أكثر من مهمة.

جدول الصلاة المجمد من [Noor Deen](https://www.noor-deen.com/prayer-times/medina/) بتوقيت Asia/Riyadh. ميزاته مواعيد جدولية، وليست أوقات إقامة أو حركة حشود. T05 يستخدم محاذاته المحسوبة الخاصة.

In [ ]:
CONFIG = tasks.default_config()
schedule = workflow.load_schedule(project)
references = project["references"]

def save_settings():
    """ينقل نفس الإعدادات القابلة للتعديل إلى أداة التقييم المحلي."""
    path = OUTPUT / "task_config.json"
    path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
    return path

CONFIG_PATH = save_settings()
display(pd.DataFrame({"task": list(CONFIG), "source": [
    "baseline_tasks.condition_forecast / forecast_value / im_forecast_value",
    "baseline_tasks.fit_hidden / hidden_value",
    "baseline_tasks.fit_hidden_future / hidden_future_value",
    "baseline_tasks.fit_missing / missing_value"]}))

# 4. FORECAST

نتنبأ بعدد الكاميرا بعد `now` بـ1–15 دقيقة. يسمح فقط بقراءات كل الكاميرات عند `now` أو قبله. المفعّل لـA–H: قالب أسبوعي بمحاذاة الصلاة + مجموعة T05 من ستة أعضاء تشترط على انحرافات الكاميرات وتتعلم ترابطها وتأخرها. لـI–M: مرجع آخر قراءة حديثة، ممزوج بوسيط آخر ثلاث قراءات إذا كانت كلها ضمن خمس دقائق.

**أين تعدّل هذه المهمة؟** `baseline_tasks.condition_forecast` و`forecast_value`، ولـI–M `im_forecast_value` التي تستدعي `im_forecast_fallback_trial.predict`. الخوارزمية الأساسية: `t05_forecast.gp.predict_recording/predict`، والقالب `t05_forecast.pipeline.make_template`. هذه الدوال تصل إلى التقييم أيضًا. `predict_forecast` و`t05_local_forecast.predict` مجرد محولين لبيانات الإنتاج والطية.

وزن النموذج وحدّ حداثة I–M ووزن الوسيط فعّالة عند التنبؤ ولا تحتاج إعادة تدريب. **لا نعرض خطوات التدريب أو ells/rank كأزرار تعديل**: إنها ضمن ملفات covariance المحملة ولن تتغير بتغيير إعداد تدريب نصي. تغيير بنية الملاءمة أو ميزات القالب يتطلب إعادة بناء أصول إنتاج وتقييم مستقلة قبل اعتمادها؛ التحميل يرفض ملفًا ناقصًا ولا يدرب بديلًا تلقائيًا.

In [ ]:
# إعدادات FORECAST الفعالة عند التنبؤ
FORECAST_CONFIG = {"model_weight": 1.0, "im_max_three_age": 5, "im_median_weight": 0.5}
assert 0 <= FORECAST_CONFIG["im_max_three_age"] <= 5  # قاعدة حداثة المرجع لا تتجاوز 5
CONFIG["forecast"] = FORECAST_CONFIG
save_settings()
forecast_targets = workflow.task_targets(project, "forecast")
display(forecast_targets[["id", "unit", "minute_of_day", "now_minute_of_day"]].head(3))
display(workflow.input_example(project, forecast_targets, "forecast"))

In [ ]:
# تحميل الأصول المجمدة وبناء القوالب؛ صفر ملاءمات covariance لـT05
forecast_models = tasks.load_forecast(DATA, ASSETS, project["frozen_config"])
print("Ensemble members:", len(forecast_models["members"]), "; fitted parameter sets: 12")

In [ ]:
# تنبؤ هذه المهمة فقط، مع تنسيق متوافق مع التقديم المقبول
forecast_predictions = workflow.forecast_output(project, forecast_models, FORECAST_CONFIG)
display(forecast_predictions[["id", "people"]].head(3))

**التقييم بعد التعديل:** من جذر المستودع:
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task forecast --config outputs/current_baseline_0800/task_config.json --output-dir outputs/evaluation/forecast_v1
```
نتائج `by_date.csv` و`by_date_camera.csv` و`verification.json` في مجلد التقييم المختار. يلزم محليًا وجود 12 ملفًا مستقلًا في `outputs/local_score/t05_fit_cache/`؛ ليست أصول الإنتاج بديلًا صحيحًا لها، ولا تأتي مع clone. الأداة تتوقف عند غيابها ولا تعيد تدريب T05. هذا هو اختبار A–H التاريخي ذو عينة ملاءمة 96 خلية وتغطية قالب محدودة، وليس تحققًا مستقلاً جديدًا. لا يوجد I–M في هذا الاختبار؛ انظر قيد تقييم I–M في دليل المساهمة أدناه.

# 5. HIDDEN

نقدّر B/G الجمعة وD السبت عند الدقيقة نفسها، مع حجب B وG معًا من ميزاتهما. يسمح HIDDEN بكامل الجدول المحرر.

المفعّل: B نموذج CatBoost (269 شجرة، عمق4، MAE) ممزوج .5 مع تقدير B السابق؛ G dropout Ridge بوزن الجمعة3 ومزج .5 مع مرجع العينة؛ D نموذج HGB السابق بميزات الحصص والصلاة. B/G يتدربان على train وtest المحرر؛ D على train فقط. غياب مانحين آنيين يعيد B/D إلى الأساس السابق؛ G يستعمل استيفاء المانحين المحدود.

**أين تعدّل؟** `baseline_tasks.fit_hidden/predict_hidden/hidden_value`، والوصفات في `hidden_bgd.fit/predict` و`hidden_catboost.features/train_model`. تغيير ميزات/إعدادات التدريب يتطلب إعادة خلية الملاءمة؛ `catboost_iterations` فعال عند الملاءمة وأوزان المزج عند التنبؤ. D يستعمل `hidden_reconstruction_trial.fit_new/make_features` دون توسيع بياناته.

إعداد0.806 محفوظ منفصلًا. لا تفعّل d_only فوق BGD. التقييم يعيد الملاءمة ببيانات الطية؛ لا يحمل نموذج الإنتاج المدرب على تواريخ التقييم.


In [ ]:
HIDDEN_CONFIG = tasks.default_config()["hidden"]
# Training settings require rerunning fitting; blend weights act at prediction.
HIDDEN_CONFIG.update(alpha=3000.0, catboost_iterations=269, catboost_weight=0.5, g_weight=0.5)
CONFIG["hidden"] = HIDDEN_CONFIG
save_settings()
hidden_targets = workflow.task_targets(project, "hidden")
display(hidden_targets[["id", "unit", "minute_of_day"]].head(3))
display(workflow.input_example(project, hidden_targets, "hidden"))


In [ ]:
hidden_models = tasks.fit_hidden(train, HIDDEN_CONFIG, schedule, released=test)
print("Production fitting sources:")
display(hidden_models["bgd"]["manifest"])


In [ ]:
hidden_raw = tasks.predict_hidden(hidden_models, hidden_targets, feeds, references, HIDDEN_CONFIG, schedule)
hidden_predictions = workflow.hidden_output(project, hidden_raw, HIDDEN_CONFIG)
display(hidden_predictions[["id", "people"]].head(3))

**التقييم بعد التعديل:**
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task hidden --config outputs/current_baseline_0800/task_config.json --output-dir outputs/evaluation/hidden_v1
```
الأداة تستبعد تاريخ التقييم بالكامل، وتستدعي `fit_hidden/hidden_value` نفسيهما، وتختبر عدم تأثير إضافة قيم للكاميرات المحجوبة. الدرجات في `by_date.csv` و`by_date_camera.csv`. لا تحتاج ذاكرة T05 لهذه المهمة.

# 6. HIDDEN-FUTURE

نتنبأ بمستقبل كاميرا مخفية بعد `now` بـ1–15 دقيقة؛ لا تاريخ للكاميرا المحجوبة ولا أي قراءة بعد `now`. المفعّل: نموذج HistGradientBoosting مباشر من تاريخ المانحين، بوزن 0.75 مع المرجع. السبت يضيف ميزات زمنية من جدول الصلاة؛ الجمعة تستخدم النسخة المباشرة العادية. لا نستخدم توقعات المانحين المستقبلية الفعلية.

**أين تعدّل هذه المهمة؟** `baseline_tasks.fit_hidden_future/hidden_future_value`. بناء المدخلات في `v2_experiments.direct_feature`، والتدريب/التنبؤ العاديان في `fit_direct/predict_hf`؛ نسخة السبت في `prayer_experiments.fit_direct_variant/predict_direct_variant`.

`model_params` يصل فعلًا إلى النموذجين عند التدريب؛ تغييره يتطلب إعادة خلية التدريب أو Run All، ولا توجد ملفات estimator محفوظة يجب استبدالها. وزن المزج واختيار نسخة السبت إعدادا تنبؤ؛ تعطيل النسخة ثم إعادة تفعيلها يستلزم إعادة خلية التدريب لبنائها.

In [ ]:
HIDDEN_FUTURE_CONFIG = {"weight": 0.75, "saturday_prayer": True,
    "model_params": {"loss": "absolute_error", "max_iter": 100, "learning_rate": 0.06,
        "max_leaf_nodes": 12, "min_samples_leaf": 40, "random_state": 27}}
CONFIG["hidden-future"] = HIDDEN_FUTURE_CONFIG
save_settings()
hidden_future_targets = workflow.task_targets(project, "hidden-future")
display(hidden_future_targets[["id", "unit", "minute_of_day", "now_minute_of_day"]].head(3))
display(workflow.input_example(project, hidden_future_targets, "hidden-future"))

In [ ]:
hidden_future_models = tasks.fit_hidden_future(train, schedule, HIDDEN_FUTURE_CONFIG)
print("Direct and Saturday prayer estimators fitted from training recordings only.")

In [ ]:
hidden_future_raw = tasks.predict_hidden_future(hidden_future_models, hidden_future_targets,
    feeds, references, HIDDEN_FUTURE_CONFIG, schedule)
hidden_future_predictions = workflow.hidden_future_output(project, hidden_future_raw,
    hidden_future_models, HIDDEN_FUTURE_CONFIG)
display(hidden_future_predictions[["id", "people"]].head(3))

**التقييم بعد التعديل:**
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task hidden-future --config outputs/current_baseline_0800/task_config.json --output-dir outputs/evaluation/hidden_future_v1
```
الملاءمة تستبعد اليوم كاملًا، ثم تستخدم الدوال المشتركة، مع اختبار استبدال جميع المشاهدات بعد `now`. تقارير التاريخ والكاميرا في مجلد التقييم. المرجع المحلي يعاد بناؤه داخل الطية؛ ليس قيم عينة الإنتاج نفسها.

# 7. MISSING

نملأ انقطاعًا داخل تسجيل لكاميرا مرئية عادةً. هذه المهمة تسمح بالمشاهدات المحررة على جانبي الفجوة، ومنها ما بعد الفجوة، ولا تحمل قيد `now` الخاص بالتنبؤ. لا نستخدم أي خلية محجوبة للهدف أو للمانحين.

المفعّل: استيفاء خطي بين الطرفين مع تصحيح من انحرافات المانحين عن خطوط طرفيهم، معاير على train. يعمل على A/B/C/E/G، وبنصف القوة على F، للفجوات ≥6 دقائق مع مانحين مناسبين. D/H وI–M والحالات غير المدعومة تبقى على مرجع الاستيفاء. تجربتا التنعيم وتعلّم شكل الفجوة مرفوضتان وغير مستخدمتين.

**أين تعدّل هذه المهمة؟** `baseline_tasks.fit_missing/missing_value`، والخوارزمية `v2_missing.fit_calibration/predict/deviations/gap_context`. تعديل المعايرة في المصدر يستلزم إعادة خلية التدريب؛ الأوزان والحد الأدنى للفجوة فعّالان عند التنبؤ ولا يحتاجان أصولًا مجمدة جديدة. لا تفعّل I–M اعتمادًا على تدريب A–H دون تحقق مستقل للنقل.

In [ ]:
MISSING_CONFIG = {"min_gap": 6, "weights": {
    "cam_A": 1.0, "cam_B": 1.0, "cam_C": 1.0, "cam_D": 0.0,
    "cam_E": 1.0, "cam_F": 0.5, "cam_G": 1.0, "cam_H": 0.0}}
CONFIG["missing"] = MISSING_CONFIG
save_settings()
missing_targets = workflow.task_targets(project, "missing")
display(missing_targets[["id", "unit", "minute_of_day"]].head(3))
display(workflow.input_example(project, missing_targets, "missing"))

In [ ]:
missing_calibration = tasks.fit_missing(train, MISSING_CONFIG)
print("Calibrated camera pairs:", len(missing_calibration))

In [ ]:
missing_raw = tasks.predict_missing(missing_calibration, missing_targets, feeds, references, MISSING_CONFIG)
missing_predictions = workflow.missing_output(project, missing_raw)
display(missing_raw[["id", "people", "used", "donors", "gap"]].head(3))

**التقييم بعد التعديل:**
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task missing --config outputs/current_baseline_0800/task_config.json --output-dir outputs/evaluation/missing_v1
```
نفس `fit_missing/missing_value`، باستبعاد تاريخ الطية بالكامل، على الحالات الـ358 الثابتة. توجد 38 فترة انقطاع: 32 كاملة الإجابات و6 جزئية بسبب فقد أصلي؛ لا نحول الدقائق المتجاورة إلى تجارب مستقلة. التقارير في مجلد التقييم، ولا يعيد الأمر تدريب T05.

# 8. جمع التوقعات والتحقق وحفظ التقديم

نجمع نتائج المهام حسب IDs العينة، ثم نفحص التغطية والتفرّد والقيم المنتهية غير السالبة. `baseline_workflow.save_combined` يحافظ على التنسيق التاريخي، ويفحص SHA-256 المقبولة. إن توفر التقديم المحمي محليًا نقارن كل قيمة أيضًا. اختلاف البصمة يوقف وضع إعادة الإنتاج، ولا يعد نجاحًا تلقائيًا.

عند تعديل مقصود، غيّر في القسم 1: `VERIFY_ACCEPTED=False`، و`OUTPUT_NAME` إلى اسم مرشح منفصل مثل `submission_missing_manual_v1.csv`، ويفضل مجلد OUTPUT مستقل. لا تغيّر البصمة المقبولة. لا يُستبدل ملف مختلف موجود مسبقًا.

In [ ]:
CONFIG_PATH = save_settings()
task_predictions = {"forecast": forecast_predictions, "hidden": hidden_predictions,
    "hidden-future": hidden_future_predictions, "missing": missing_predictions}
verification = workflow.save_combined(project, task_predictions, CONFIG, OUTPUT,
    verify_accepted=VERIFY_ACCEPTED, filename=OUTPUT_NAME, protected=protected, started=started)
assert verification["ids"] == 4457 and verification["valid_nonnegative"]
if VERIFY_ACCEPTED:
    assert verification["sha256_matches_accepted"]
display(pd.DataFrame(verification.get("by_task", {})).T)
print(json.dumps(verification, indent=2))
print("Settings for local evaluation:", CONFIG_PATH)

### مساهمة زميل وتشغيل التقييم

1. أنشئ فرعًا: `git switch -c task/hidden-update`، وعدّل مهمة واحدة في المصدر المذكور أو إعداداتها.
2. اختر OUTPUT واسم مرشح منفصلين و`VERIFY_ACCEPTED=False` للتغيير المقصود؛ لا تغير بصمة الأساس لإخفاء الفرق.
3. Restart Kernel ثم Run All، ثم قيّم **بنفس task_config.json** في مجلد جديد، وراجع التغطية والكاميرات والنوافذ قبل commit يدوي.
4. لا يوجد رفع أوcommit تلقائي. ضمّن المصدر والدفتر والإعدادات والأصول المتتبعة؛ البيانات والنواتج خارج Git.

الربط بالتقييم: `evaluate_baseline_task.py` و`reliable_evaluation.py` يستدعيان وصفات المهام المشتركة. HIDDEN يعيد الملاءمة لكل طية دون تحميل نماذج الإنتاج. لإعداد الإتاحة أضف `--hidden-protocol release_matched` إلى أمر HIDDEN؛ لا تجمع درجاته مع الصارم. أمر local_score التاريخي أرشيفي وقد يلائم GP إذا غابت ذاكرته، فلا تستخدمه لهذا التعديل.

أصول تقييم FORECAST المستقلة موجودة في `models/local_evaluation/`، لكن provenance الإنشاء القديم غير مكتمل؛ يلزم الإقرار الموثق ولا يوجد refit تلقائي. لا تستبدلها بأصول الإنتاج. I–M له بروكسي منفصل متداخل لا يكشف إجابات المسابقة. المقياس المحلي خاص بالمشروع، والتواريخ فُحصت مرارًا وليست تحققًا جديدًا مستقلًا.

التعليمات الحالية وأوامر النسخة النظيفة واستعادة0.806: `docs/teammate_baseline_0800.md`؛ الإعداد السابق `configs/baseline_0806.json`.
